# Storm and Zillow combined data

$Price_{T+1} = a + B*Range + C*TotalDamage + D*Duration + E*NationalChange$

## Data Loading helper classes

In [68]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pytest
import seaborn as sns
import datetime
import numpy as np

In [150]:
noaa_2025_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260116.csv.gz"
noaa_2024_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2024_c20260116.csv.gz"
noaa_2023_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2023_c20260116.csv.gz"

noaa_2025_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2025_c20260116.csv.gz"
noaa_2024_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2024_c20260116.csv.gz"
noaa_2023_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2023_c20260116.csv.gz"

zillow_county_url = "https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv?t=1769612738"
zillow_msa_url = "https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv?t=1769612738"

In [147]:
class NOAAStormParser:
    def __init__(self, details_url, locations_url):
        details_df = pd.read_csv(details_url)
        locations_df = pd.read_csv(locations_url)
        self.df = pd.merge(locations_df, details_df, on="EVENT_ID", how="left")
        self.clean()
        
    def clean(self):
        self.parse_dates()
        self.parse_damages()
        self.parse_storm_duration()

    def parse_dates(self):
        self.df["YEARMONTH"] = pd.to_datetime(self.df["YEARMONTH"], format='%Y%m') + pd.offsets.MonthEnd(0)
        
    def parse_storm_duration(self):
        self.df["BEGIN_DATE_TIME"] = pd.to_datetime(self.df["BEGIN_DATE_TIME"])
        self.df["END_DATE_TIME"] = pd.to_datetime(self.df["END_DATE_TIME"])
        duration_delta = self.df["END_DATE_TIME"] - self.df["BEGIN_DATE_TIME"]
        self.df["DURATION_HOURS"] = duration_delta.dt.total_seconds() / 3600.0
        
    def parse_damages(self):
        self.df['DAMAGE_PROPERTY'] = self.df["DAMAGE_PROPERTY"].apply(self.parse_numeric_string)
        self.df['DAMAGE_CROPS'] = self.df["DAMAGE_CROPS"].apply(self.parse_numeric_string)
        self.df['TOTAL_DAMAGE'] = self.df["DAMAGE_CROPS"] + self.df["DAMAGE_PROPERTY"]

    def parse_numeric_string(self, rawstring):
        if pd.isna(rawstring):
            return 0

        rawstring = str(rawstring) 
        
        if "K" in rawstring:
            value = rawstring.replace("K","")
            return float(value) * 1000
        if "M" in rawstring:
            value = rawstring.replace("M","")
            return float(value) * 1_000_000
        if "B" in rawstring:
            value = rawstring.replace("B","")
            return float(value) * 1_000_000_000
        return float(rawstring)


In [100]:
class ZillowDataParser():
    DTYPES = {
        'RegionID': 'Int64',
        'SizeRank': 'Int64',
        'RegionName': 'object',
        'RegionType': 'object',
        'StateName': 'object',
        'State': 'object',
        'Metro': 'object',
        'StateCodeFIPS': 'Int64',
        'MunicipalCodeFIPS': 'Int64'
    }
    
    META_DATA_COLS = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
                      'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS',
                      'latitude', 'longitude']
    
    def __init__(self, regional_url, national_url):
        self.download_data(regional_url, national_url)
        
    def download_data(self, regional_url, national_url):
        regional_data_raw = pd.read_csv(regional_url)
        national_data_raw = pd.read_csv(national_url)
        national = national_data_raw.loc[national_data_raw["RegionName"] == "United States"]
        self.df = pd.concat([regional_data_raw, national_data_raw],sort=False, ignore_index=True)
        self.df = self.df.astype(self.DTYPES)
        all_columns = self.df.columns.tolist() + [col for col in self.META_DATA_COLS if col not in self.df.columns]
        self.df = self.df.reindex(columns=all_columns)
        
    def fips_time_series(self, fips: tuple = ()):
        """
        Returns DataFrame with one column of time series data
        """
        regional = self.df.loc[(self.df['StateCodeFIPS'] == fips[0]) & (self.df['MunicipalCodeFIPS'] == fips[1])]
        
        if regional.empty:
            return pd.DataFrame()
        
        regional_data = regional.loc[:, self.date_cols()].T
        
        region_name = regional['RegionName'].iloc[0]
        new_columns = pd.MultiIndex.from_tuples(
            [(region_name, fips[0], fips[1])],
            names=['Region', 'StateFIPS', 'CountyFIPS']
        )
        
        regional_data.columns = new_columns
        regional_data.index = pd.to_datetime(regional_data.index)
        return regional_data
    
    def normalized_from_date(self, start_date, months_back=0):
        """
        Returns a DataFrame that is scaled to 100 as of the start_date
        """
        start_date = pd.to_datetime(start_date)
        
        
        if months_back > 0:
            window_start_date = start_date - pd.DateOffset(months=months_back)
        else:
            window_start_date = start_date
        
        all_dates = self.date_cols()
        result_dates = [date for date in all_dates if pd.to_datetime(date) >= window_start_date]
        
        if not result_dates:
            return pd.DataFrame()
        ref_col = min(result_dates, key=lambda x: abs(pd.to_datetime(x) - start_date))
        
        result_cols = self.META_DATA_COLS + result_dates        
        normalized_df = self.df[result_cols].copy()
        baseline = normalized_df[ref_col] / 100.0
        normalized_df[result_dates] = normalized_df[result_dates].div(baseline, axis=0)
        return normalized_df

    def change_after_event(self, event_date, fips):
        """
        Returns a DataFrame with the normalized index for a set of region
        1, 2, 3, 6, 12 months after an event data
        """
        event_date = datetime(event_date)
        norm_df = self.normalized_from_date(event_date, months_back=24)
        country_mask = norm_df["RegionType"] == "country"
        fips_mask = (norm_df['StateCodeFIPS'] == fips[0]) & (norm_df['MunicipalCodeFIPS'] == fips[1])
        norm_df = norm_df[fips_mask | country_mask]
        available_dates = [col for col in norm_df.columns if col not in self.META_DATA_COLS]
        event_date = pd.to_datetime(event_date)
        intervals = list(range(1,13))
        results = norm_df[self.META_DATA_COLS].copy()
        
        for m in intervals:
            plus_date = event_date + pd.DateOffset(months=m)
            closest_date = min(available_dates, key=lambda x: abs(pd.to_datetime(x) - plus_date))
            
            if abs(pd.to_datetime(closest_date) - plus_date).days < 15:
                 results[f'plus_{m}_mon'] = norm_df[closest_date]
            else:
                 results[f'plus_{m}_mon'] = np.nan
            
            minus_date = event_date - pd.DateOffset(months=m)
            closest_minus = min(available_dates, key=lambda x: abs(pd.to_datetime(x) - minus_date))
            if abs(pd.to_datetime(closest_minus) - minus_date).days < 15:
                results[f'minus_{m}_mon'] = norm_df[closest_minus]
            else:
                results[f'minus_{m}_mon'] = np.nan
        
        results.reset_index(inplace=True, drop=True)
        return results

    def latest_date(self):
        return self.date_cols()[-1]

    def date_cols(self):
        dates = [col for col in self.df.columns if col not in self.META_DATA_COLS]
        dates = sorted(dates, key=pd.to_datetime)
        return list(dates)
    
    def get_monthly_panel(self):
        """
        Transforms the Wide Zillow data into a Long format panel.
        """
        date_cols = [col for col in self.df.columns if col not in self.META_DATA_COLS]

        long_df = self.df.melt(
            id_vars=self.META_DATA_COLS, 
            value_vars=date_cols,
            var_name='Date', 
            value_name='Index_Value'
        )
        

        long_df['Date'] = pd.to_datetime(long_df['Date'])
        long_df = long_df.sort_values(['StateCodeFIPS', 'MunicipalCodeFIPS', 'Date'])
        long_df['Regional_Pct_Change'] = long_df.groupby(['StateCodeFIPS', 'MunicipalCodeFIPS'])['Index_Value'].pct_change()

        national_df = long_df[long_df['RegionName'] == 'United States'].copy()
        national_df = national_df.sort_values('Date')
        national_df['Regional_Pct_Change'] = national_df['Index_Value'].pct_change()

        national_df = national_df[['Date', 'Index_Value', 'Regional_Pct_Change']]
        national_df = national_df.rename(columns={
            'Index_Value': 'National_Index',
            'Regional_Pct_Change': 'National_Pct_Change'
        })

        final_df = pd.merge(long_df, national_df, on='Date', how='left')

        final_df = final_df.set_index(['StateCodeFIPS', 'MunicipalCodeFIPS', 'Date'])
        
        return final_df

## Load  Data

In [151]:
noaa2025 = NOAAStormParser(noaa_2025_details, noaa_2025_locations) 
noaa2024 = NOAAStormParser(noaa_2024_details, noaa_2024_locations) 
noaa2023 = NOAAStormParser(noaa_2023_details, noaa_2023_locations) 

In [113]:
zillow = ZillowDataParser(zillow_county_url, zillow_msa_url)

In [114]:
zillow.df.describe()

,RegionID,SizeRank,StateCodeFIPS,MunicipalCodeFIPS,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,2000-06-30,...,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,latitude,longitude
count,3968.000000,3968.000000,3073.000000,3073.000000,1465.000000,1468.000000,1471.000000,1475.000000,1480.000000,1483.000000,...,3.968000e+03,3.968000e+03,3.968000e+03,3.968000e+03,3.967000e+03,3.968000e+03,3.968000e+03,3.968000e+03,0.0,0.0
mean,94422.441028,1328.516129,30.270420,103.426944,109086.734949,109281.604623,109437.574709,110061.588849,110721.489693,111324.180075,...,2.645894e+05,2.641817e+05,2.640125e+05,2.638937e+05,2.639281e+05,2.645452e+05,2.652859e+05,2.662720e+05,NaN,NaN
std,175804.256207,945.214094,15.072376,107.816180,53963.775283,54117.997242,54341.796456,54957.051354,55572.965375,56234.440451,...,1.682290e+05,1.679394e+05,1.675511e+05,1.673361e+05,1.672145e+05,1.682018e+05,1.689697e+05,1.697155e+05,NaN,NaN
min,66.000000,0.000000,1.000000,1.000000,26035.323550,26099.406623,26181.452638,26357.464955,26579.437292,26808.349726,...,4.372812e+04,4.266551e+04,4.182409e+04,4.091705e+04,3.985900e+04,3.846328e+04,3.684678e+04,3.517614e+04,NaN,NaN
25%,1108.750000,505.000000,18.000000,35.000000,74644.682173,74833.977784,74931.380179,75331.532921,75644.576975,76019.837897,...,1.648016e+05,1.641707e+05,1.641868e+05,1.642539e+05,1.644563e+05,1.645439e+05,1.649672e+05,1.656379e+05,NaN,NaN
50%,2146.500000,1105.500000,29.000000,79.000000,96242.907215,96340.163894,96311.358293,96429.652697,97201.018270,97471.395587,...,2.220464e+05,2.218427e+05,2.223179e+05,2.222101e+05,2.228967e+05,2.235721e+05,2.245144e+05,2.255136e+05,NaN,NaN
75%,3180.500000,2150.250000,45.000000,133.000000,128166.893724,128271.779080,128382.350770,129009.999532,130024.236993,130383.477703,...,3.147711e+05,3.144703e+05,3.148403e+05,3.148607e+05,3.149215e+05,3.159256e+05,3.166209e+05,3.175451e+05,NaN,NaN
max,753929.000000,3214.000000,56.000000,840.000000,656736.508148,658869.745073,662020.864070,668994.231789,678146.821451,687351.857838,...,2.901127e+06,2.897887e+06,2.890448e+06,2.877436e+06,2.880154e+06,2.898973e+06,2.927875e+06,2.958707e+06,NaN,NaN


## Combine Data
- Each location, pull the event data from Zillow for regional and national changes

In [152]:
combined_noaa_df = pd.concat([noaa2025.df, noaa2024.df, noaa2023.df], ignore_index=True)

In [165]:
combined_noaa_df.columns

Index(['YEARMONTH', 'EPISODE_ID_x', 'EVENT_ID', 'LOCATION_INDEX', 'RANGE',
       'AZIMUTH', 'LOCATION', 'LATITUDE', 'LONGITUDE', 'LAT2', 'LON2',
       'BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH',
       'END_DAY', 'END_TIME', 'EPISODE_ID_y', 'STATE', 'STATE_FIPS', 'YEAR',
       'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO',
       'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME', 'INJURIES_DIRECT',
       'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT',
       'DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'SOURCE', 'MAGNITUDE',
       'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY', 'TOR_F_SCALE',
       'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE',
       'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 'BEGIN_RANGE',
       'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH',
       'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON',
       'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE', 'TOTAL_DAMAGE',
   

In [159]:
# Aggregate the weather events by month and location
agg_rules = {
    'EVENT_ID': 'count',
    'DURATION_HOURS': 'sum',
    'TOTAL_DAMAGE': 'sum',
    
}
monthly_summary = combined_noaa_df.groupby(['STATE_FIPS', 'CZ_FIPS', 'YEARMONTH']).agg(agg_rules).reset_index()

# Convert duration to days
monthly_summary['DURATION_HOURS'] = monthly_summary['DURATION_HOURS'] / 24

# Convert damage to billions USD
monthly_summary['TOTAL_DAMAGE'] = monthly_summary['TOTAL_DAMAGE'] / 1_000_000_000

monthly_summary = monthly_summary.rename(columns={
    'EVENT_ID': 'EVENT_COUNT',
    'DURATION_HOURS': 'TOTAL_DURATION_DAYS',
    'TOTAL_DAMAGE': 'TOTAL_DAMAGE_BILLIONS'
})

In [123]:
def merge_with_lag(housing_df, storm_df, lag_months=1):
    lagged_storms = storm_df.copy()

    rename_map = {
        'STATE_FIPS': 'StateCodeFIPS',
        'CZ_FIPS': 'MunicipalCodeFIPS',
        'YEARMONTH': 'Date'
    }
    lagged_storms = lagged_storms.rename(columns=rename_map)

    if not pd.api.types.is_datetime64_any_dtype(lagged_storms['Date']):
        lagged_storms['Date'] = pd.to_datetime(lagged_storms['Date'])

    lagged_storms['Date'] = lagged_storms['Date'] + pd.DateOffset(months=lag_months)

    if 'Date' not in housing_df.columns:
        housing_temp = housing_df.reset_index()
    else:
        housing_temp = housing_df.copy()

    merged_df = pd.merge(
        housing_temp,
        lagged_storms,
        on=['StateCodeFIPS', 'MunicipalCodeFIPS', 'Date'],
        how='inner' 
    )

    return merged_df

In [154]:
df = merge_with_lag(zillow.get_monthly_panel(), monthly_summary, lag_months=6)

/opt/conda/lib/python3.9/site-packages/pandas/core/reshape/merge.py:1203: RuntimeWarning: invalid value encountered in cast
  if not (lk == lk.astype(rk.dtype))[~np.isnan(lk)].all():


In [163]:
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np


features = ['National_Pct_Change', 'TOTAL_DAMAGE_BILLIONS']
target = 'Regional_Pct_Change'

regression_df = df.dropna(subset=features + [target]).copy()
regression_df['Regional_Pct_Change'] = regression_df['Regional_Pct_Change'] * 100
regression_df['National_Pct_Change'] = regression_df['National_Pct_Change'] * 100

X = regression_df[features]
y = regression_df[target]

model = LinearRegression()
model.fit(X, y)

sklearn_summary(model, X, y, features)

Variable                  | Coef         | Std Err      | t-stat     | P>|t|     
--------------------------------------------------------------------------------
Intercept                 | -0.029229    | 0.010478     | -2.7897    | 0.0053     **
National_Pct_Change       | 1.070521     | 0.050171     | 21.3374    | 0.0000     ***
TOTAL_DAMAGE_BILLIONS     | -0.173175    | 0.111185     | -1.5575    | 0.1194     
--------------------------------------------------------------------------------
Residual Std Error: 0.6058 on 9976 degrees of freedom
R-squared:          0.0438


In [132]:
import numpy as np
from scipy import stats

def sklearn_summary(model, X, y, feature_names):
    """
    Replicates the R/Statsmodels summary table for a Scikit-Learn model.
    Calculates Standard Errors, t-stats, and p-values manually.
    """
    # 1. Prepare Data
    # Ensure X is a numpy array
    X_arr = np.array(X)
    y_arr = np.array(y)
    
    # Append a column of 1s for the Intercept (Standard OLS Math)
    X_with_intercept = np.c_[np.ones(len(X_arr)), X_arr]
    
    # 2. Calculate Residuals & Variance
    predictions = model.predict(X)
    residuals = y_arr - predictions
    RSS = np.sum(residuals ** 2)
    
    # Degrees of Freedom: n - k (k includes intercept)
    n = len(X_arr)
    k = len(feature_names) + 1 
    dof = n - k
    
    # Mean Squared Error (Variance of residuals)
    MSE = RSS / dof
    
    # 3. Covariance Matrix: MSE * inv(X.T @ X)
    # This tells us the variance of each coefficient
    try:
        cov_matrix = MSE * np.linalg.inv(X_with_intercept.T @ X_with_intercept)
    except np.linalg.LinAlgError:
        print("Error: Matrix is Singular (Collinear features). Cannot compute Standard Errors.")
        return

    # Standard Errors are the square root of the diagonal
    std_errors = np.sqrt(np.diag(cov_matrix))
    
    # 4. Coefficients (Intercept + Slopes)
    params = np.append(model.intercept_, model.coef_)
    
    # 5. T-Statistics & P-Values
    t_stats = params / std_errors
    p_values = [2 * (1 - stats.t.cdf(np.abs(t), dof)) for t in t_stats]
    
    # 6. Print the Table
    print(f"{'Variable':<25} | {'Coef':<12} | {'Std Err':<12} | {'t-stat':<10} | {'P>|t|':<10}")
    print("-" * 80)
    
    # Add 'Intercept' to names
    all_names = ['Intercept'] + feature_names
    
    for name, coef, se, t, p in zip(all_names, params, std_errors, t_stats, p_values):
        star = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{name:<25} | {coef:<12.6f} | {se:<12.6f} | {t:<10.4f} | {p:<10.4f} {star}")
    
    print("-" * 80)
    print(f"Residual Std Error: {np.sqrt(MSE):.4f} on {dof} degrees of freedom")
    print(f"R-squared:          {model.score(X, y):.4f}")